# 03 — Similarity & Clustering: Cell Lines, Drugs, and Pairs

**Goal (diagnostic, not modeling):** understand how similar/redundant cell lines and
drugs are to each other by modality, to anticipate how "hard" the LCO/LDO/LTO splits
actually are. This complements DrEval's Normalized PCC metric — it doesn't replace it;
the splits + normalized metric are what structurally guard against Simpson's paradox.

1. Highly variable feature (HVG) selection, RNA and protein separately
2. Cell line clustering per modality + agreement between modalities (Adjusted Rand Index)
3. Drug structural similarity + clustering (Tanimoto on Morgan fingerprints)
4. Pair-level response heterogeneity: within-cluster vs across-cluster variance

> Column names below are assumed from `SESSION_MEMORY.md` — verify against actual
> headers in the CONFIG cell before running.

## Config

In [ ]:
from pathlib import Path

DATA_DIR = Path("../data/GDSC2")
RESULTS_DIR = Path("../results")
FIGURES_DIR = RESULTS_DIR / "figures"
PROCESSED_DIR = Path("../data/processed")

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

COL_CELL_LINE = "cell_line_name"
COL_DRUG = "drug_name"
COL_SMILES = "smiles"
COL_IC50 = "LN_IC50"
COL_CELLOSAURUS = "cellosaurus_id"
COL_TISSUE = "tissue"

N_HVG = 2000
N_CLUSTERS_CELL = 8
N_CLUSTERS_DRUG = 8
RANDOM_STATE = 42

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
from sklearn.preprocessing import StandardScaler
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import squareform
from rdkit import Chem
from rdkit.Chem import AllChem, DataStructs

sns.set_theme(style="whitegrid")

## Load data (same three-way overlap logic as notebook 02)

In [ ]:
rna = pd.read_csv(DATA_DIR / "gene_expression.csv", index_col=0)
protein = pd.read_csv(DATA_DIR / "proteomics.csv", index_col=0)
cell_lines = pd.read_csv(DATA_DIR / "cell_line_names.csv")
gdsc = pd.read_csv(DATA_DIR / "GDSC2.csv")
drug_smiles = pd.read_csv(DATA_DIR / "drug_smiles.csv")


def three_way_overlap(
    rna_df: pd.DataFrame,
    protein_df: pd.DataFrame,
    gdsc_df: pd.DataFrame,
    mapping_df: pd.DataFrame,
) -> list[str]:
    """Cellosaurus IDs present in RNA, protein, and GDSC2 response data."""
    name_to_id = dict(zip(mapping_df[COL_CELL_LINE], mapping_df[COL_CELLOSAURUS]))
    gdsc_ids = set(gdsc_df[COL_CELL_LINE].map(name_to_id).dropna())
    return sorted(set(rna_df.index) & set(protein_df.index) & gdsc_ids)


common_ids = three_way_overlap(rna, protein, gdsc, cell_lines)
tissue_lookup = cell_lines.set_index(COL_CELLOSAURUS)[COL_TISSUE]
len(common_ids)

## 1. Highly variable feature selection (HVG)

In [ ]:
def select_hvg(
    df: pd.DataFrame, cell_ids: list[str], n_top: int = 2000, n_bins: int = 20
) -> list[str]:
    """Top variable features, normalizing dispersion within mean-expression bins
    (Seurat-style) so highly-expressed features don't dominate by raw variance alone."""
    sub = df.loc[cell_ids]
    mean = sub.mean(axis=0)
    var = sub.var(axis=0)
    dispersion = var / mean.replace(0, np.nan)

    bins = pd.qcut(mean.rank(method="first"), n_bins, labels=False)
    z_dispersion = dispersion.groupby(bins).transform(lambda x: (x - x.mean()) / x.std(ddof=0))
    return z_dispersion.sort_values(ascending=False).head(n_top).index.tolist()


hvg_rna = select_hvg(rna, common_ids, N_HVG)
hvg_protein = select_hvg(protein, common_ids, N_HVG)
len(hvg_rna), len(hvg_protein), len(set(hvg_rna) & set(hvg_protein))

## 2. Cell line clustering — RNA vs Protein

In [ ]:
def cluster_cell_lines(
    df: pd.DataFrame, cell_ids: list[str], features: list[str], n_clusters: int
) -> tuple[np.ndarray, np.ndarray]:
    """Z-score, PCA to 20 components, KMeans. Returns (pca_coords, cluster_labels)."""
    X = StandardScaler().fit_transform(df.loc[cell_ids, features])
    pcs = PCA(n_components=20, random_state=RANDOM_STATE).fit_transform(X)
    labels = KMeans(n_clusters=n_clusters, random_state=RANDOM_STATE, n_init=10).fit_predict(pcs)
    return pcs, labels


rna_pcs, rna_clusters = cluster_cell_lines(rna, common_ids, hvg_rna, N_CLUSTERS_CELL)
protein_pcs, protein_clusters = cluster_cell_lines(protein, common_ids, hvg_protein, N_CLUSTERS_CELL)

ari = adjusted_rand_score(rna_clusters, protein_clusters)
print(f"Adjusted Rand Index (RNA clusters vs Protein clusters): {ari:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, pcs, clusters, title in [
    (axes[0], rna_pcs, rna_clusters, "RNA-based clusters"),
    (axes[1], protein_pcs, protein_clusters, "Protein-based clusters"),
]:
    ax.scatter(pcs[:, 0], pcs[:, 1], c=clusters, cmap="tab10", s=12, alpha=0.8)
    ax.set_title(title)
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
fig.suptitle(f"Cell line clustering by modality (ARI = {ari:.3f})")
fig.savefig(FIGURES_DIR / "fig2_cell_line_clusters_rna_vs_protein.png", dpi=200, bbox_inches="tight")

In [ ]:
contingency = pd.crosstab(
    pd.Series(rna_clusters, name="RNA cluster"),
    pd.Series(protein_clusters, name="Protein cluster"),
)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(contingency, annot=True, fmt="d", cmap="Blues", ax=ax)
ax.set_title("RNA vs Protein cluster agreement")
fig.savefig(FIGURES_DIR / "fig3_cluster_agreement_heatmap.png", dpi=200, bbox_inches="tight")

## 3. Drug structural similarity

In [ ]:
def smiles_to_fingerprint(smiles: str, radius: int = 2, n_bits: int = 2048):
    """Morgan fingerprint as an RDKit ExplicitBitVect, or None if SMILES is invalid."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)


def drug_tanimoto_matrix(
    smiles_df: pd.DataFrame, drug_col: str, smiles_col: str
) -> tuple[pd.DataFrame, list[str]]:
    """Pairwise Tanimoto similarity matrix for all drugs with a valid SMILES string."""
    fps, names = [], []
    for _, row in smiles_df.iterrows():
        fp = smiles_to_fingerprint(row[smiles_col])
        if fp is not None:
            fps.append(fp)
            names.append(row[drug_col])

    n = len(fps)
    sim = np.zeros((n, n))
    for i in range(n):
        sim[i] = DataStructs.BulkTanimotoSimilarity(fps[i], fps)
    return pd.DataFrame(sim, index=names, columns=names), names


drug_sim, drug_names = drug_tanimoto_matrix(drug_smiles, COL_DRUG, COL_SMILES)
drug_sim.shape

In [ ]:
drug_dist = 1 - drug_sim.values
np.fill_diagonal(drug_dist, 0)
condensed = squareform(drug_dist, checks=False)
drug_linkage = linkage(condensed, method="average")

fig, ax = plt.subplots(figsize=(12, 5))
dendrogram(drug_linkage, labels=drug_names, no_labels=True, ax=ax)
ax.set_title("Drug structural similarity (Tanimoto, average linkage)")
ax.set_xlabel("Drugs")
ax.set_ylabel("Distance (1 - Tanimoto)")
fig.savefig(FIGURES_DIR / "fig4_drug_dendrogram.png", dpi=200, bbox_inches="tight")

## 4. Pair-level heterogeneity — within vs across clusters

In [ ]:
drug_cluster_labels = fcluster(drug_linkage, N_CLUSTERS_DRUG, criterion="maxclust")
drug_cluster_map = dict(zip(drug_names, drug_cluster_labels))
cell_cluster_map = dict(zip(common_ids, rna_clusters))  # RNA-based, treated as primary modality

name_to_id = dict(zip(cell_lines[COL_CELL_LINE], cell_lines[COL_CELLOSAURUS]))
pairs = gdsc[gdsc[COL_DRUG].isin(drug_cluster_map)].copy()
pairs[COL_CELLOSAURUS] = pairs[COL_CELL_LINE].map(name_to_id)
pairs = pairs[pairs[COL_CELLOSAURUS].isin(cell_cluster_map)]

pairs["cell_cluster"] = pairs[COL_CELLOSAURUS].map(cell_cluster_map)
pairs["drug_cluster"] = pairs[COL_DRUG].map(drug_cluster_map)
pairs.shape

In [ ]:
def within_vs_across_variance(df: pd.DataFrame, value_col: str, group_col: str) -> pd.DataFrame:
    """Mean within-cluster variance vs overall (across-cluster) variance of value_col, per drug."""
    rows = []
    for drug, sub in df.groupby(COL_DRUG):
        if sub[group_col].nunique() < 2:
            continue
        within = sub.groupby(group_col)[value_col].var().mean()
        across = sub[value_col].var()
        rows.append({"drug": drug, "within_cluster_var": within, "across_cluster_var": across})
    return pd.DataFrame(rows)


variance_df = within_vs_across_variance(pairs, COL_IC50, "cell_cluster")
variance_df["ratio"] = variance_df["within_cluster_var"] / variance_df["across_cluster_var"]
variance_df.to_csv(PROCESSED_DIR / "pair_variance_within_vs_across_cell_clusters.csv", index=False)
variance_df["ratio"].median()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ax.hist(variance_df["ratio"].dropna(), bins=30, color="#55A868", edgecolor="white")
ax.axvline(1.0, color="black", linestyle="--", label="within = across")
ax.set_xlabel("within-cluster variance / across-cluster variance")
ax.set_ylabel("Number of drugs")
ax.set_title("Response heterogeneity within vs across cell-line clusters")
ax.legend()
fig.savefig(FIGURES_DIR / "fig5_within_vs_across_variance.png", dpi=200, bbox_inches="tight")

## Notes & next steps

- **Ratio well below 1** → cell-line clusters explain real response structure; LCO/LTO
  are testing something real, not just noise.
- **Ratio near 1** → clusters (as currently defined) don't capture response-relevant
  structure; worth revisiting `N_CLUSTERS_CELL` or the HVG feature set.
- The ARI between RNA-based and protein-based clusters tells us how much *new*
  structure protein actually contributes at the cell-line level — low ARI is a good
  sign for the multimodal hypothesis, high ARI suggests redundancy.
- In the baseline notebooks (05/06), we can stratify LCO/LTO results by whether a held-out
  cell line was cluster-isolated or cluster-adjacent to the training set — a direct,
  data-grounded probe of the Simpson's-paradox concern from the DrEval paper (Fig. 3).